# Analiza e rezultateve për Trigger Arc TSP

Ky notebook lexon results_optimized.csv dhe best_results.csv, krahason koston fillestare me rezultatin më të mirë dhe vizaton përmirësimet. Nuk e kopjon kodin e zgjidhësit.

Në Kaggle, shtoje repository-n si Dataset (përfshi dosjen results/) te Add Input. Nëse Kaggle nuk është kërkesë e profesorit, notebook-u mund të përdoret lokalisht.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

roots = [Path.cwd(), Path('/kaggle/input')]
best_files = []
for root in roots:
    if root.exists():
        best_files.extend(root.rglob('best_results.csv'))
if not best_files:
    raise FileNotFoundError('Nuk u gjet best_results.csv. Shto dataset-in te Kaggle Add Input ose hape notebook-un nga repository.')

best_path = best_files[0]
results_dir = best_path.parent
best = pd.read_csv(best_path)
optimized_path = results_dir / 'results_optimized.csv'
optimized = pd.read_csv(optimized_path) if optimized_path.exists() else pd.DataFrame()
display(best.head(20))


In [ ]:
comp = best.dropna(subset=['baseline_cost', 'best_cost']).copy()
for col in ['baseline_cost', 'best_cost', 'improvement_percent']:
    comp[col] = pd.to_numeric(comp[col], errors='coerce')
display(comp[['instance','baseline_cost','best_cost','improvement_percent','best_seed','status']])

if not comp.empty:
    ax = comp.set_index('instance')[['baseline_cost','best_cost']].plot(kind='bar', figsize=(12,5), color=['#9aa4b2','#2b8aef'])
    ax.set_title('Kostoja fillestare dhe kostoja më e mirë')
    ax.set_ylabel('Kostoja')
    ax.set_xlabel('Instanca')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:
if not optimized.empty:
    optimized['cost'] = pd.to_numeric(optimized['cost'], errors='coerce')
    sa = optimized[optimized['algorithm'].str.contains('insertion+swap', regex=False, na=False)]
    display(sa.groupby('instance').agg(best_cost=('cost','min'), runs=('seed','count'), average_seconds=('seconds','mean')).reset_index())
else:
    print('CSV-ja e rezultateve të optimizuara nuk u gjet pranë best_results.csv.')
